[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/cours/seance3_cours.ipynb)

# Séance 2.3 — Agréger et croiser plusieurs tables

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- filtrer sur plusieurs conditions sans se noyer dans les parenthèses
- classer et extraire un top 5 en une commande
- répondre à « combien par ... ? » avec `groupby`
- calculer plusieurs indicateurs d'un coup avec `agg`
- rassembler trois fichiers en une seule table avec `merge`
- croiser deux dimensions avec un tableau croisé

## Retour à la question de départ

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous savez maintenant charger et nettoyer. Et pourtant vous ne pouvez toujours
pas répondre — pour une raison très simple :

**`ventes.csv` ne contient pas le pays.** Il contient un `client_id`. Le pays
est dans `clients.csv`.

C'est la situation normale en entreprise : l'information est **répartie entre
plusieurs fichiers**, et la réponse naît de leur croisement. C'est l'objet de
cette séance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]
print(ventes.shape, clients.shape, produits.shape)

## 1. Filtrer sur plusieurs conditions

### Empiler avec `and` / `or`

In [ ]:
grosses = ventes.query("qte >= 50 and prix < 2")
print(len(grosses), "lignes : beaucoup d'unites, prix unitaire faible")

Vous verrez souvent l'autre écriture :

```python
ventes[(ventes["qte"] >= 50) & (ventes["prix"] < 2)]
```

Comptez : **4 crochets, 4 parenthèses, 2 fois le nom de la table** — contre
une paire de guillemets avec `.query()`. Reconnaissez-la quand vous la croisez
dans du code trouvé en ligne, mais écrivez `.query()`.

### Une liste de valeurs — `in`

In [ ]:
# Attention aux guillemets : doubles a l'exterieur, simples a l'interieur
ue = clients.query("pays in ['France', 'Allemagne', 'Belgique']")
print(len(ue), "clients dans ces trois pays")

### Un intervalle

In [ ]:
moyennes = ventes.query("50 <= qte <= 100")
print(len(moyennes), "lignes")

## 2. Trier et extraire un top

In [ ]:
# ascending=False : du plus grand au plus petit
ventes.sort_values("ca", ascending=False).head(3)[["prod_id", "qte", "prix", "ca"]]

In [ ]:
# nlargest fait la meme chose en plus court
ventes.nlargest(3, "ca")[["prod_id", "qte", "ca"]]

## 3. `groupby` — la commande la plus utile de tout le bloc

`groupby` répond à toutes les questions de la forme **« combien par ... ? »**.

Trois temps, toujours les mêmes :

1. **Découper** les lignes en paquets selon une colonne
2. **Calculer** un indicateur dans chaque paquet
3. **Recoller** les résultats en un tableau

In [ ]:
# "Combien de chiffre d'affaires par client ?"
ca_client = ventes.groupby("client_id")["ca"].sum()

ca_client.nlargest(5).round(2)

Un client pèse à lui seul **143 825 €**. Gardez ce chiffre en tête, on y
reviendra en séance 2.4.

### Plusieurs indicateurs d'un coup — `agg`

In [ ]:
resume = ventes.groupby("client_id").agg(
    ca=("ca", "sum"),              # total depense
    nb_lignes=("cmd_id", "count"), # nombre de LIGNES
    nb_cmd=("cmd_id", "nunique"),  # nombre de COMMANDES distinctes
)
resume.nlargest(3, "ca").round(2)

La syntaxe se lit : `nom_voulu=("colonne_source", "operation")`.

> ⚠️ **`count` vs `nunique` — l'erreur classique.**
> `count` compte les **lignes**. `nunique` compte les **valeurs distinctes**.
> Une commande de 30 articles occupe 30 lignes mais reste **une** commande.
> Regardez l'écart entre `nb_lignes` et `nb_cmd` ci-dessus : confondre les
> deux, c'est diviser son panier moyen par 20.

Les opérations disponibles : `"sum"`, `"mean"`, `"median"`, `"min"`, `"max"`,
`"count"`, `"nunique"`, `"std"`.

## 4. `merge` — rassembler les fichiers

C'est l'équivalent du `RECHERCHEV` d'Excel, en beaucoup plus sûr.

Les deux tables ont une colonne en commun : `client_id`. `merge` s'en sert
pour aller chercher, pour chaque vente, les informations du client
correspondant.

In [ ]:
avant = len(ventes)
vc = ventes.merge(clients, on="client_id")

# LE reflexe : verifier qu'on n'a ni perdu ni duplique de lignes
print(avant, "->", len(vc))

> ⚠️ **Ne sautez jamais cette vérification.** Si la clé de jointure n'est pas
> unique dans la table de droite, `merge` **duplique** des lignes sans rien
> dire. Vos totaux deviennent faux et rien ne vous alerte. Deux nombres
> affichés, une seconde de lecture, et vous êtes tranquille.

`vc` contient maintenant les colonnes des deux tables :

In [ ]:
vc[["client_id", "pays", "segment", "qte", "prix", "ca"]].head(3)

### Et enfin, la réponse à la question du bloc

In [ ]:
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=False)

ca_pays.head(5).round(2)

Le Royaume-Uni domine — c'est le marché domestique, sans surprise.

**Mais regardez l'Irlande : 261 205 €, deuxième marché du groupe.** Combien
de clients irlandais y a-t-il ?

In [ ]:
vc.query("pays == 'Irlande'")["client_id"].nunique()

**Deux.** Deux clients pèsent 22,7 % du chiffre d'affaires total.

Gardez cette anomalie de côté : c'est le point de départ de l'étude de cas de
la séance 2.4. Une moyenne par pays ne veut rien dire quand deux clients font
le marché.

### Enchaîner les jointures

In [ ]:
# On ajoute maintenant les informations produit
complet = vc.merge(produits, on="prod_id")
print(len(complet), "lignes,", complet.shape[1], "colonnes")

complet.groupby("categorie")["ca"].sum().nlargest(3).round(2)

## 5. Croiser deux dimensions

`groupby` répond à « par pays ». Et « par pays **et** par segment » ?

In [ ]:
top4 = ca_pays.head(4).index

vc.query("pays in @top4").pivot_table(
    values="ca", index="pays", columns="segment", aggfunc="sum"
).round(0)

> 💡 Le `@top4` dans `query()` veut dire « va chercher la variable Python
> nommée `top4` ». Pratique pour ne pas retaper une liste.

Deux cases sont vides pour l'Irlande. Ce n'est pas un bug : ses deux clients
sont tous deux classés « premium », il n'y a donc **aucune** ligne
« occasionnel » ou « standard » à additionner. **Une case vide dans un
tableau croisé est une information**, pas une erreur.

### Compter plutôt que sommer — `crosstab`

In [ ]:
pd.crosstab(complet.query("pays in @top4")["pays"],
            complet.query("pays in @top4")["categorie"])

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| filtrer sur une condition | `df.query("prix > 10")` |
| combiner des conditions | `df.query("prix > 10 and pays == 'France'")` |
| une liste de valeurs | `df.query("pays in ['France', 'Belgique']")` |
| un intervalle | `df.query("50 <= qte <= 100")` |
| trier | `df.sort_values("ca", ascending=False)` |
| les 5 plus grands | `df.nlargest(5, "ca")` |
| total par groupe | `df.groupby("pays")["ca"].sum()` |
| plusieurs indicateurs | `df.groupby("pays").agg(ca=("ca", "sum"), n=("cmd_id", "nunique"))` |
| joindre deux tables | `a.merge(b, on="client_id")` |
| croiser deux dimensions | `df.pivot_table(values="ca", index="pays", columns="segment", aggfunc="sum")` |
| compter des croisements | `pd.crosstab(df["pays"], df["categorie"])` |

## Les deux erreurs à ne jamais commettre

1. **Faire un `merge` sans vérifier le nombre de lignes avant et après.**
   Un `merge` peut silencieusement dupliquer ou faire disparaître des lignes.
   `print(len(a), "->", len(fusion))` : une seconde, et vous dormez tranquille.

2. **Confondre `count` et `nunique`.** `count` compte les lignes,
   `nunique` compte les valeurs distinctes. Une commande de 30 articles,
   c'est 30 lignes mais **une** commande.